# 🌦️ Two-Tool Weather Agent: Current Conditions vs. Forecast

**Goal:** We want to answer two different kinds of questions:

1. *"I am living in New Delhi. I am planning to go outside now. Do I need cold or warm clothes?"*
   -> needs **current** weather
2. *"Should I carry an umbrella in New Delhi tomorrow?"*
   -> needs a **forecast**, not current conditions

## 0. Setup


In [ ]:
# Uncomment to install
# !pip install -q "langchain>=1.0" langchain-openai requests

In [9]:
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", "langchain-openai", ]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-openai version: 1.2.2


In [1]:
import getpass
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key:  ········


## 1. Tool 1 — Current Weather

Same tool as before: current temperature, "feels like," and conditions for a city.


In [2]:
import requests
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """
    Get the CURRENT weather right now for a given city.
    Use this for questions about what to wear or do right now / today.
    Input should be a city name, e.g. 'New Delhi'.
    """
    url = f"https://wttr.in/{city}?format=j1"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json()

    current = data["current_condition"][0]
    temp_c = current["temp_C"]
    feels_like_c = current["FeelsLikeC"]
    description = current["weatherDesc"][0]["value"]

    return (
        f"Current weather in {city}: {temp_c}°C, "
        f"feels like {feels_like_c}°C, conditions: {description}."
    )

## 2. Tool 2 — Weather Forecast (new)

This tool answers a **different kind of question**: not "what's it like right now" but
"what will it be like in N days." response already includes a 3-day forecast
in its `weather` array, so we reuse the same API — just a different part of the response.

Note the docstring: it explicitly says this is for **future** days, so the model has a clear
signal on when to prefer this tool over `get_weather`.


In [3]:
@tool
def get_forecast(city: str, days_ahead: int = 1) -> str:
    """Get the weather FORECAST for a given city, for a future day.
    Use this for questions about tomorrow, this weekend, or any future day —
    NOT for questions about right now (use get_weather for that).

    Args:
        city: city name, e.g. 'New Delhi'.
        days_ahead: how many days from today (0 = today, 1 = tomorrow, 2 = day after). Max 2.
    """
    days_ahead = max(0, min(days_ahead, 2))  # wttr.in's free tier gives today + 2 days

    url = f"https://wttr.in/{city}?format=j1"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json()

    day = data["weather"][days_ahead]
    date = day["date"]
    max_temp = day["maxtempC"]
    min_temp = day["mintempC"]

    # Average the hourly chance-of-rain readings for a simple daily estimate
    rain_chances = [int(hour["chanceofrain"]) for hour in day["hourly"]]
    avg_rain_chance = sum(rain_chances) // len(rain_chances)

    return (
        f"Forecast for {city} on {date} ({days_ahead} day(s) ahead): "
        f"high {max_temp}°C, low {min_temp}°C, "
        f"average chance of rain: {avg_rain_chance}%."
    )

In [4]:
# Quick manual tests of both tools
print(get_weather.invoke("New Delhi"))
print(get_forecast.invoke({"city": "New Delhi", "days_ahead": 1}))

Current weather in New Delhi: 29°C, feels like 36°C, conditions: Light Rain With Thunderstorm, Rain With Thunderstorm.
Forecast for New Delhi on 2026-07-06 (1 day(s) ahead): high 38°C, low 31°C, average chance of rain: 10%.


## 3. Build the Agent with Both Tools

Now the agent has **two** tools with clearly different purposes. The system prompt doesn't
need to tell it which one to use for which question — that's exactly the reasoning we want
the model to do on its own, guided only by each tool's docstring.


In [4]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = [get_weather, get_forecast]

SYSTEM_PROMPT = (
    "You are a helpful weather assistant. Always check real weather data before "
    "answering questions about clothing, umbrellas, or outdoor plans. Choose "
    "between the current-weather tool and the forecast tool based on whether "
    "the question is about right now or a future day. Clearly explain your "
    "recommendation using the actual data you observed."
)

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

## 4. A Helper to Run a Question and Show Which Tool Was Chosen

Since we now have two tools, the interesting thing to observe is **which tool the agent
picks** for each question — that's the **reasoning** step we're demonstrating.


In [5]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})

    print(f"QUESTION: {question}\n")

    for msg in result["messages"]:
        if type(msg).__name__ == "AIMessage" and getattr(msg, "tool_calls", None):
            for call in msg.tool_calls:
                print(f"  -> Agent chose tool: {call['name']}  args={call['args']}")
        elif type(msg).__name__ == "ToolMessage":
            print(f"  -> Observation: {msg.content}")

    print(f"\nFINAL ANSWER:\n{result['messages'][-1].content}")
    print("\n" + "=" * 70 + "\n")
    return result

## 5. Test Both Question Types

Watch which tool gets picked for each — this is the agent reasoning about **which data it
actually needs**, not just calling whatever tool exists.


In [7]:
_ = ask(
    "I am living in New Delhi. I am planning to go outside right now. "
    "Do I need cold clothes or warm clothes?"
)

QUESTION: I am living in New Delhi. I am planning to go outside right now. Do I need cold clothes or warm clothes?

  -> Agent chose tool: get_weather  args={'city': 'New Delhi'}
  -> Observation: Current weather in New Delhi: 29°C, feels like 36°C, conditions: Light Rain With Thunderstorm, Rain With Thunderstorm.

FINAL ANSWER:
The current weather in New Delhi is 29°C, but it feels like 36°C due to humidity and conditions of light rain with thunderstorms. Given this information, I recommend wearing light, breathable clothing, but also consider bringing a light rain jacket or umbrella to stay dry.




In [8]:
_ = ask(
    "I am living in New Delhi. Should I carry an umbrella tomorrow?"
)

QUESTION: I am living in New Delhi. Should I carry an umbrella tomorrow?

  -> Agent chose tool: get_forecast  args={'city': 'New Delhi', 'days_ahead': 1}
  -> Observation: Forecast for New Delhi on 2026-07-06 (1 day(s) ahead): high 38°C, low 31°C, average chance of rain: 10%.

FINAL ANSWER:
The weather forecast for New Delhi tomorrow indicates a high of 38°C and a low of 31°C, with only a 10% chance of rain. Given this low probability of rain, you likely won't need to carry an umbrella. Enjoy your day!




# Memory

In [6]:
from langgraph.checkpoint.memory import InMemorySaver

# A checkpointer gives the agent memory: it stores the message history
# per "thread_id", so the same conversation can span multiple .invoke() calls.
checkpointer = InMemorySaver()

memory_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

# Every conversation needs a thread_id — this is how the agent knows
# which conversation's memory to load. Different thread_ids = separate,
# isolated conversations (e.g. different users).
config = {"configurable": {"thread_id": "demo-conversation-1"}}

# Turn 1: mention the city explicitly
result1 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "I live in New Delhi. Do I need warm or cold clothes right now?"}]},
    config=config,
)
print("Turn 1:", result1["messages"][-1].content)

# Turn 2: a follow-up that does NOT repeat the city.
# Without memory, the agent wouldn't know which city we're asking about.
# With the checkpointer, it recalls "New Delhi" from Turn 1 automatically.
result2 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "What about tomorrow — should I carry an umbrella?"}]},
    config=config,
)
print("\nTurn 2:", result2["messages"][-1].content)

Turn 1: Right now in New Delhi, the temperature is 28°C, but it feels like 27°C. The weather conditions include mist and rain with thunderstorms. 

Given this information, I recommend wearing light, breathable clothing since it's warm, but also consider bringing a raincoat or umbrella due to the rain and thunderstorms.

Turn 2: Tomorrow in New Delhi, the forecast indicates a high of 38°C and a low of 31°C, with an average chance of rain at 10%. 

Given the low probability of rain, it seems unlikely that you'll need an umbrella. However, it will be quite hot, so make sure to stay hydrated and wear light clothing.


## 6. What This Demonstrates

- **Tool selection as reasoning**: with two tools available, the agent has to infer from the
  *question's wording* ("right now" vs. "tomorrow") which tool actually answers it — this is a
  more convincing demonstration of reasoning than a single-tool agent, where there's no real
  choice to make.
- **Docstrings are the interface**: notice we never wrote "if the question contains the word
  'tomorrow', use get_forecast." The tool docstrings alone (mentioning "right now" vs. "future
  day") were enough for the model to route correctly. This is worth calling out explicitly —
  good tool docstrings do a lot of the reasoning work for you.
- **Same underlying loop, more branching**: it's still think -> act -> observe -> think -> answer,
  just with a real decision point now (which tool?) instead of only one option.

### Try it yourself
- Ask an ambiguous question like *"What's the weather going to be like?"* (no "today" or
  "tomorrow") and see which tool the agent defaults to — a good discussion prompt on
  prompt/docstring ambiguity.
- Ask a question needing **both** tools in one turn, e.g. *"Compare today's weather to
  tomorrow's in New Delhi"* — you should see two tool calls in the trace.
